# Anime Recommender — Build & Save Pickle
Run this on Kaggle (T4x2). Downloads `anime_model.pkl` at the end.

In [ ]:
import numpy as np
import pandas as pd
import re
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sentence_transformers import SentenceTransformer


In [ ]:
df = pd.read_csv('/kaggle/input/datasets/pratyushsahu02/animedataset/anime_dataset_v4.csv')
print(df.shape)
df.head()


In [ ]:
# ── Episode cleaning ─────────────────────────────────────────────────────────
df['Episodes'] = df['Episodes'].astype('Int64')
df.loc[df['Status'] == 'NOT_YET_RELEASED', 'Episodes'] =     df.loc[df['Status'] == 'NOT_YET_RELEASED', 'Episodes'].fillna(0)
df.loc[df['Status'] == 'FINISHED', 'Episodes'] =     df.loc[df['Status'] == 'FINISHED', 'Episodes'].fillna(1)
df['Is_Releasing'] = (df['Status'] == 'RELEASING').astype(int)
df['Episodes'] = df['Episodes'].fillna(-1)

# ── Genre / Tag / Description missing flags ───────────────────────────────────
df['Genres_Missing']     = df['Genres'].isna().astype(int)
df['Genres']             = df['Genres'].fillna('Unknown')
df['Tags_Missing']       = df['Tags'].isna().astype(int)
df['Tags']               = df['Tags'].fillna('Unknown')
df['Description_Missing']= (df['Description'] == '').astype(int)
df['Description']        = df['Description'].fillna('')

# ── Lowercase + replace commas ────────────────────────────────────────────────
for col in ['Title', 'Genres', 'Tags', 'Studio', 'Description', 'Format']:
    df[col] = df[col].astype(str).str.lower()
df['Genres'] = df['Genres'].str.replace(',', ' ')
df['Tags']   = df['Tags'].str.replace(',', ' ')

print('Preprocessing done')


In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

for col in ['Title', 'Genres', 'Tags', 'Studio', 'Description']:
    df[col] = df[col].apply(clean_text)

df['Content'] = (
    (df['Genres'] + ' ') * 3 +
    (df['Tags']   + ' ') * 4 +
    df['Description'] + ' ' +
    df['Format']  + ' ' +
    df['Studio']
)
print('Content column built')


In [ ]:
# ── TF-IDF ────────────────────────────────────────────────────────────────────
tfidf = TfidfVectorizer(stop_words='english', max_features=10000, ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(df['Content'])
print('TF-IDF shape:', tfidf_matrix.shape)


In [ ]:
# ── Sentence Embeddings (T4x2 — batch_size=256 is fine) ──────────────────────
model = SentenceTransformer('all-mpnet-base-v2', device='cuda')
embeddings = model.encode(
    df['Content'].tolist(),
    batch_size=256,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)
print('Embeddings shape:', embeddings.shape)


In [ ]:
# ── Normalize popularity & score ──────────────────────────────────────────────
scaler = MinMaxScaler()
df[['pop_norm', 'score_norm']] = scaler.fit_transform(
    df[['Popularity', 'AverageScore']]
)

# ── Title → index lookup ──────────────────────────────────────────────────────
indices = pd.Series(df.index, index=df['Title']).drop_duplicates()
print('Scaler + indices ready')


In [ ]:
# ── Save everything into one pickle ──────────────────────────────────────────
# Columns kept in df: only what Streamlit needs (saves ~30% file size)
KEEP_COLS = ['Title', 'AverageScore', 'Popularity', 'Genres',
             'Format', 'pop_norm', 'score_norm']

payload = {
    'df':          df[KEEP_COLS].reset_index(drop=True),
    'tfidf_matrix': tfidf_matrix,   # scipy sparse — stays small
    'embeddings':   embeddings,      # numpy float32 array
    'indices':      indices,
}

PICKLE_PATH = '/kaggle/working/anime_model.pkl'
with open(PICKLE_PATH, 'wb') as f:
    pickle.dump(payload, f, protocol=5)   # protocol=5 → fastest, Python 3.8+

import os
size_mb = os.path.getsize(PICKLE_PATH) / 1e6
print(f'Saved → {PICKLE_PATH}  ({size_mb:.1f} MB)')


In [ ]:
# ── Quick sanity check ────────────────────────────────────────────────────────
with open(PICKLE_PATH, 'rb') as f:
    check = pickle.load(f)

print('Keys:', list(check.keys()))
print('df shape:', check['df'].shape)
print('embeddings shape:', check['embeddings'].shape)
print('tfidf_matrix shape:', check['tfidf_matrix'].shape)
print('indices length:', len(check['indices']))
print('\nAll good — download anime_model.pkl from Kaggle Output tab.')
